In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📊 Financial Report Analyst

## What We're Building

An assistant that can answer questions over **annual reports** and **earnings call
transcripts** — including questions about tables, figures, and financial numbers.

## The Challenge: Tables in Documents

Financial reports are full of tables:
```
Revenue       | 2023    | 2024    | Change
Product Sales | $12.5B  | $14.2B  | +13.6%
Services      | $8.3B   | $9.1B   | +9.6%
```

Standard RAG treats tables as plain text and often breaks them across chunks,
losing the row/column structure. Our approach:

1. **Detect tables** in the text
2. **Keep tables whole** (don't split mid-table)
3. **Add table context** as metadata for better retrieval

## Stack
- **LangChain** — orchestration
- **Table-aware text splitter** — custom splitting that preserves tables
- **ChromaDB** — vector store
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

import re
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Financial Data

A realistic (simulated) annual report excerpt and earnings call transcript.

In [ ]:
annual_report = """ACME CORP — ANNUAL REPORT 2024

LETTER TO SHAREHOLDERS

Dear Shareholders,

Fiscal year 2024 was a transformative year for Acme Corp. We achieved record
revenue of $22.8 billion, representing 14% year-over-year growth. Our strategic
investments in AI and cloud infrastructure are paying dividends, with our Cloud
Services segment growing 28% and now representing 42% of total revenue.

We expanded into three new international markets (Brazil, India, and Germany)
and acquired DataFlow Inc. for $1.2 billion, adding 2,000 engineers and a
market-leading data pipeline product to our portfolio.

FINANCIAL HIGHLIGHTS

Consolidated Income Statement (in millions USD):

| Metric                  | FY 2024    | FY 2023    | Change   |
|-------------------------|------------|------------|----------|
| Total Revenue           | $22,800    | $20,000    | +14.0%   |
| Product Revenue         | $13,224    | $12,400    | +6.6%    |
| Cloud Services Revenue  | $9,576     | $7,600     | +26.0%   |
| Cost of Revenue         | $11,400    | $10,200    | +11.8%   |
| Gross Profit            | $11,400    | $9,800     | +16.3%   |
| Gross Margin            | 50.0%      | 49.0%      | +1.0pp   |
| Operating Expenses      | $6,840     | $6,200     | +10.3%   |
| Operating Income        | $4,560     | $3,600     | +26.7%   |
| Operating Margin        | 20.0%      | 18.0%      | +2.0pp   |
| Net Income              | $3,648     | $2,800     | +30.3%   |
| Earnings Per Share      | $7.30      | $5.60      | +30.4%   |

Balance Sheet Highlights:

| Item                    | Dec 2024   | Dec 2023   |
|-------------------------|------------|------------|
| Cash & Equivalents      | $8,500     | $7,200     |
| Total Assets            | $45,000    | $38,000    |
| Total Debt              | $12,000    | $10,000    |
| Shareholders' Equity    | $25,000    | $21,000    |
| Debt-to-Equity Ratio    | 0.48       | 0.48       |

SEGMENT PERFORMANCE

Cloud Services:
Revenue grew 26% to $9.58B driven by enterprise migrations and AI workloads.
Customer retention rate was 95%. Average contract value increased 18% to $450K.
We added 1,200 new enterprise customers, bringing the total to 8,500.

Product Division:
Revenue grew 6.6% to $13.2B. Growth was driven by the new X-Series hardware
platform. Unit shipments declined 3% but average selling price increased 10%
due to favorable product mix shift toward premium tiers.

OUTLOOK FOR 2025

We expect FY 2025 revenue of $25.5-26.5 billion (12-16% growth).
Cloud Services is expected to cross $12 billion. We plan $3.5 billion in
capital expenditure, primarily for data center expansion in Asia-Pacific.
We anticipate operating margin expansion to 21-22%.
"""

earnings_call = """ACME CORP — Q4 2024 EARNINGS CALL TRANSCRIPT

PREPARED REMARKS — CEO Jane Smith:

Good afternoon. Q4 was an outstanding quarter, capping a record year. Let me
highlight the key takeaways.

Q4 revenue came in at $6.2 billion, up 16% year-over-year and above our
guidance range of $5.8-6.0 billion. Cloud Services was the star, growing 32%
in Q4 alone, driven by three large government contracts and strong enterprise
demand for our AI inference platform.

The DataFlow acquisition closed in September and contributed $180 million in
Q4 revenue. Integration is ahead of schedule — we've already achieved $40
million in cost synergies.

PREPARED REMARKS — CFO Tom Johnson:

Q4 gross margin expanded 150 basis points to 51.5%, driven by cloud scale
efficiencies and the ongoing hardware mix shift. Operating expenses grew 8%,
well below revenue growth, demonstrating operating leverage.

Free cash flow was $1.8 billion in Q4 and $5.2 billion for the full year.
We returned $2.0 billion to shareholders through dividends and buybacks.

Q4 Financial Summary:

| Metric              | Q4 2024  | Q4 2023  | Change  |
|---------------------|----------|----------|---------|  
| Revenue             | $6,200   | $5,350   | +15.9%  |
| Gross Margin        | 51.5%    | 50.0%    | +1.5pp  |
| Operating Income    | $1,364   | $1,070   | +27.5%  |
| Net Income          | $1,091   | $834     | +30.8%  |
| EPS (diluted)       | $2.18    | $1.67    | +30.5%  |
| Free Cash Flow      | $1,800   | $1,400   | +28.6%  |

ANALYST Q&A:

Analyst (Morgan Stanley): Can you speak to the AI demand trends? What
percentage of cloud revenue is AI-related?

CEO: Great question. AI workloads now represent approximately 35% of our
cloud revenue, up from 20% a year ago. We're seeing strong demand for both
training and inference. Our AI inference platform launched in Q2 and already
has 400 enterprise customers.

Analyst (Goldman Sachs): What's the customer concentration risk? How
dependent are you on the top accounts?

CFO: Our top 10 customers represent 18% of total revenue, down from 22%
two years ago. No single customer exceeds 3%. We've been deliberately
diversifying our customer base.

Analyst (JPMorgan): Can you elaborate on the 2025 CapEx plans?

CEO: We're investing $3.5 billion, with $2.2 billion allocated to three new
data centers: Singapore, Mumbai, and São Paulo. These will be operational
by Q3 2025 and support our 2026 growth targets. The remaining $1.3 billion
is for GPU cluster expansion and network upgrades at existing facilities.
"""

print(f"Annual report: {len(annual_report.split())} words")
print(f"Earnings call: {len(earnings_call.split())} words")

## Step 3 — Table-Aware Text Splitting

The key insight: **never split a table across chunks.** We detect
markdown tables and treat each table as an atomic unit.

In [ ]:
def extract_tables_and_text(content: str) -> list[dict]:
    """
    Split content into text sections and table sections.
    Tables are kept whole, text is split normally.
    """
    # Regex to detect markdown tables (lines starting with |)
    lines = content.split("\n")
    segments = []
    current_text = []
    current_table = []
    in_table = False
    
    for line in lines:
        is_table_line = line.strip().startswith("|") and "|" in line.strip()[1:]
        is_separator = bool(re.match(r'^\|[\s-|]+\|$', line.strip()))
        
        if is_table_line or is_separator:
            if not in_table and current_text:
                # Save pending text
                text = "\n".join(current_text).strip()
                if text:
                    segments.append({"type": "text", "content": text})
                current_text = []
            in_table = True
            current_table.append(line)
        else:
            if in_table and current_table:
                # Save completed table
                table = "\n".join(current_table).strip()
                if table:
                    segments.append({"type": "table", "content": table})
                current_table = []
            in_table = False
            current_text.append(line)
    
    # Save remaining
    if current_text:
        text = "\n".join(current_text).strip()
        if text:
            segments.append({"type": "text", "content": text})
    if current_table:
        table = "\n".join(current_table).strip()
        if table:
            segments.append({"type": "table", "content": table})
    
    return segments


# Test on annual report
segments = extract_tables_and_text(annual_report)
print(f"Found {len(segments)} segments:")
for s in segments:
    preview = s['content'][:60].replace('\n', ' ')
    print(f"  [{s['type']:5s}] {preview}...")

In [ ]:
def create_table_aware_chunks(
    content: str,
    source: str,
    doc_type: str,
) -> list[Document]:
    """
    Split financial text into chunks, keeping tables whole.
    Text sections are split normally; tables remain intact.
    """
    segments = extract_tables_and_text(content)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". "],
    )
    
    docs = []
    for segment in segments:
        if segment["type"] == "table":
            # Tables stay whole — never split
            doc = Document(
                page_content=segment["content"],
                metadata={
                    "source": source,
                    "doc_type": doc_type,
                    "content_type": "table",
                },
            )
            docs.append(doc)
        else:
            # Text is split normally
            text_doc = Document(
                page_content=segment["content"],
                metadata={
                    "source": source,
                    "doc_type": doc_type,
                    "content_type": "text",
                },
            )
            split_docs = text_splitter.split_documents([text_doc])
            docs.extend(split_docs)
    
    return docs


# Create chunks from both documents
report_chunks = create_table_aware_chunks(annual_report, "Annual_Report_2024.pdf", "annual_report")
call_chunks = create_table_aware_chunks(earnings_call, "Q4_2024_Earnings_Call.pdf", "earnings_call")

all_chunks = report_chunks + call_chunks

# Statistics
tables = [c for c in all_chunks if c.metadata["content_type"] == "table"]
texts = [c for c in all_chunks if c.metadata["content_type"] == "text"]
print(f"Total chunks: {len(all_chunks)} ({len(tables)} tables, {len(texts)} text)")
print(f"\nTable chunks (kept intact):")
for t in tables:
    rows = t.page_content.count("\n") + 1
    print(f"  📊 {rows} rows from {t.metadata['source']}")

## Step 4 — Index and Build Retriever

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="financial_analyst",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"Indexed {vectorstore._collection.count()} chunks")

## Step 5 — Financial Q&A Chain

The prompt is tuned for financial analysis — it knows how to read tables
and work with financial metrics.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

finance_prompt = ChatPromptTemplate.from_template(
    """You are a financial analyst assistant. Answer questions about the company's
financial performance using the provided context from annual reports and earnings calls.

GUIDELINES:
- Cite specific numbers, percentages, and dollar amounts from the data
- When referencing tables, read rows and columns accurately
- Provide YoY comparisons when data is available
- Distinguish between reported metrics and forward guidance
- Note the source document (annual report vs earnings call)
- If data for the exact question isn't available, state what IS available

Context:
{context}

Question: {question}

Analysis:"""
)


def ask_finance(question: str) -> None:
    """Ask a question about the financial data."""
    print(f"📊 {question}")
    print("-" * 60)
    
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata['source']} | Type: {d.metadata['content_type']}]\n"
        f"{d.page_content}"
        for d in docs
    )
    
    chain = finance_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    print(f"\n{answer}")
    print("=" * 60 + "\n")


print("Financial analyst ready!")

## Step 6 — Ask Financial Questions!

In [ ]:
ask_finance("What was the total revenue for FY 2024 and how did it grow?")

In [ ]:
ask_finance("Break down the revenue by segment. Which segment grew faster?")

In [ ]:
ask_finance("What is the company's debt-to-equity ratio and cash position?")

In [ ]:
ask_finance("What is the 2025 revenue guidance and CapEx plan?")

In [ ]:
# Question requiring data from BOTH documents
ask_finance("What percentage of cloud revenue is AI-related and how is it trending?")

In [ ]:
ask_finance("What is the customer concentration risk? How dependent is the company on top accounts?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Table detection** | Identifies markdown tables in financial text |
| **Table-aware splitting** | Keeps tables whole, splits only prose |
| **Content type metadata** | Tags chunks as `table` or `text` for context |
| **Financial prompt** | Instructs LLM to cite numbers and read tables accurately |
| **Multi-document Q&A** | Searches across annual report AND earnings call |

## 🔧 Customization Ideas

- **PDF table extraction**: Use `camelot` or `tabula-py` for real PDFs
- **Historical comparisons**: Index multiple years of reports
- **Calculation verification**: Ask LLM to verify computed ratios against tables
- **Visualization**: Generate charts from extracted financial data